In [1]:
import tensorflow as tf

In [2]:
daily_sales_number = [21, 22, -108, 31, -1, 32, 34, 31]

In [3]:
#We want to build a tf Dataset out of this now from a python list

tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_number)
tf_dataset

<TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>

In [4]:
for sales in tf_dataset:
    print(sales )  #The individual element here is a tensor

tf.Tensor(21, shape=(), dtype=int32)
tf.Tensor(22, shape=(), dtype=int32)
tf.Tensor(-108, shape=(), dtype=int32)
tf.Tensor(31, shape=(), dtype=int32)
tf.Tensor(-1, shape=(), dtype=int32)
tf.Tensor(32, shape=(), dtype=int32)
tf.Tensor(34, shape=(), dtype=int32)
tf.Tensor(31, shape=(), dtype=int32)


In [5]:
for sales in tf_dataset:
    print(sales.numpy())    #This turns it into a numpy array

21
22
-108
31
-1
32
34
31


In [6]:
for sales in tf_dataset.take(3):
    print(sales.numpy())

21
22
-108


In [7]:
#Sales number can't be negative, so we have to filter the data. In pipeline, get
#rid of invalid data points

tf_dataset = tf_dataset.filter(lambda x : x > 0)
for sales in tf_dataset:
    print(sales.numpy())

21
22
31
32
34
31


In [8]:
tf_dataset = tf_dataset.map(lambda x : x * 72)
for sales in tf_dataset:
    print(sales.numpy())

1512
1584
2232
2304
2448
2232


In [9]:
tf_dataset = tf_dataset.shuffle(3)
for sales in tf_dataset:
    print(sales.numpy())

2232
2304
1512
2448
2232
1584


In [10]:
for sales_batch in tf_dataset.batch(2):
    print(sales_batch.numpy())

[1512 2304]
[1584 2232]
[2232 2448]


In [11]:
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_number)

#Now single line. This is the crux of tensorflow input pipeline
tf_dataset = tf_dataset.filter(lambda x : x > 0).map(lambda y : y* 72).shuffle(3).batch(2)
for sales in tf_dataset:
    print(sales.numpy())

[1512 2304]
[2448 1584]
[2232 2232]


In [12]:
images_ds = tf.data.Dataset.list_files('images/*/*', shuffle = False)

for file in images_ds.take(5):
    print(file.numpy())

b'images\\cat\\pexels-catscoming-1543793.jpg'
b'images\\cat\\pexels-kmerriman-20787.jpg'
b'images\\cat\\pexels-kowalievska-1170986.jpg'
b'images\\cat\\pexels-pixabay-104827.jpg'
b'images\\cat\\pexels-pixabay-45201.jpg'


In [13]:
images_ds = images_ds.shuffle(200)

for file in images_ds.take(3):
    print(file.numpy())

b'images\\dog\\pexels-valeriya-1805164.jpg'
b'images\\dog\\pexels-katlovessteve-551628.jpg'
b'images\\dog\\pexels-lucasandrade-4681107.jpg'


In [14]:
class_names = ['cat', 'dog']

In [15]:
image_count = len(images_ds)
image_count

12

In [16]:
train_size = int(image_count * 0.8)

train_ds = images_ds.take(train_size)
test_ds = images_ds.skip(train_size)

In [17]:
len(train_ds)

9

In [18]:
len(test_ds)

3

In [19]:
s = 'images\\cat\\pexels-kowalievska-1170986.jpg'
s.split('\\')

['images', 'cat', 'pexels-kowalievska-1170986.jpg']

In [20]:
s.split('\\')[-2]

'cat'

In [21]:
import os

In [22]:
def get_label(file_path):
    return tf.strings.split(file_path, os.path.sep)[-2]

In [23]:
#The 'x' part

def process_image(file_path):
    label = get_label(file_path)
    img = tf.io.read_file(file_path)    #API for reading the file
    img = tf.image.decode_jpeg(img)     #decode the JPEG
    img = tf.image.resize(img, [128, 128])
    
    return img, label

In [24]:
for t in train_ds.take(4):
    print(t)                #Okay so this is a tensor

tf.Tensor(b'images\\dog\\pexels-valeriya-1805164.jpg', shape=(), dtype=string)
tf.Tensor(b'images\\cat\\pexels-wojciech-kumpicki-1084687-2071882.jpg', shape=(), dtype=string)
tf.Tensor(b'images\\dog\\pexels-lucasandrade-4681107.jpg', shape=(), dtype=string)
tf.Tensor(b'images\\dog\\pexels-svetozar-milashevich-99573-1490908.jpg', shape=(), dtype=string)


In [25]:
for t in train_ds.take(4):
    print(t.numpy())

b'images\\dog\\pexels-pixabay-220938.jpg'
b'images\\cat\\pexels-pixabay-104827.jpg'
b'images\\dog\\pexels-svetozar-milashevich-99573-1490908.jpg'
b'images\\cat\\pexels-pixabay-45201.jpg'


In [26]:
for label in train_ds.map(get_label):
    print(label)        #We got the 'y' part here

tf.Tensor(b'dog', shape=(), dtype=string)
tf.Tensor(b'cat', shape=(), dtype=string)
tf.Tensor(b'cat', shape=(), dtype=string)
tf.Tensor(b'cat', shape=(), dtype=string)
tf.Tensor(b'cat', shape=(), dtype=string)
tf.Tensor(b'dog', shape=(), dtype=string)
tf.Tensor(b'cat', shape=(), dtype=string)
tf.Tensor(b'dog', shape=(), dtype=string)
tf.Tensor(b'dog', shape=(), dtype=string)


In [27]:
#Until now, train_ds only contains the file paths

In [28]:
train_ds = train_ds.map(process_image)
for img, label in train_ds.take(3):
    print("Image: ", img)
    print("Label: ", label)        

Image:  tf.Tensor(
[[[0.00000000e+00 2.35457916e+01 2.55457916e+01]
  [0.00000000e+00 1.38483124e+01 2.08483124e+01]
  [0.00000000e+00 1.43404694e+01 2.13404694e+01]
  ...
  [0.00000000e+00 1.64233856e+01 1.94233856e+01]
  [0.00000000e+00 1.73828125e+01 1.83828125e+01]
  [3.91940002e+01 3.42643127e+01 2.86688232e+01]]

 [[8.36593628e-01 2.76477814e+01 3.06477814e+01]
  [3.01177979e+00 1.80117798e+01 2.50117798e+01]
  [1.20249939e+00 1.23534393e+01 1.93534393e+01]
  ...
  [0.00000000e+00 2.23456268e+01 2.43456268e+01]
  [0.00000000e+00 1.52949066e+01 1.82949066e+01]
  [2.93490601e+00 3.49049683e+01 3.19049683e+01]]

 [[9.57031250e-01 2.57452087e+01 2.97452087e+01]
  [3.84368896e-02 1.23692169e+01 1.93692169e+01]
  [0.00000000e+00 1.16687469e+01 1.96687469e+01]
  ...
  [0.00000000e+00 2.32466125e+01 2.72466125e+01]
  [0.00000000e+00 2.36360931e+01 2.56360931e+01]
  [9.23385620e-01 3.57731781e+01 3.77731781e+01]]

 ...

 [[9.15495300e+01 8.65495300e+01 6.45495300e+01]
  [1.12345932e+02 9.

In [29]:
#We got out numpy array. Now we need to scale it : [0, 1]

def scale(image, label):
    return image/255, label

In [30]:
train_ds = train_ds.map(scale)

for image, label in train_ds.take(5):
    print("Image: ", image.numpy()[0][0])
    print("Label: ", label.numpy())

Image:  [0.8980392  0.92941177 0.9411765 ]
Label:  b'cat'
Image:  [0. 0. 0.]
Label:  b'cat'
Image:  [0.2098652 0.2608456 0.3353554]
Label:  b'dog'
Image:  [0.63437474 0.677512   0.7010414 ]
Label:  b'cat'
Image:  [0.1        0.07254902 0.04901961]
Label:  b'cat'
